# Сжатие в Lakehouse: ORC + агрегации

Второй ноутбук. Заполняет столбцы: `формат`, `Размер, ГБ`, `Файлов`, `Время записи, с`, `Агрегация первая, сек`, `Агрегация вторая и следующие, сек`.

Что делаем:
1. Локальный бенчмарк **Parquet vs ORC** (PyArrow) — размер и скорость записи.
2. Iceberg-таблицы в Trino: **Parquet** и **ORC** с разными кодеками — размер, кол-во файлов, время вставки.
3. **Агрегации** в Trino: первый запуск vs повторные (эффект кэша).
4. Итоговая таблица под твою форму.

## 1. Импорты и клиенты

In [1]:
import io, os, time, json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.orc as orc
import pyarrow.fs as pafs
import requests
import boto3
from IPython.display import display

S3_ENDPOINT = "http://localhost:9000"
S3_KEY = S3_MINIO_USER = "minioadmin"
S3_SECRET = "minioadmin"
TRINO_URL = "http://localhost:8080"

fs = pafs.S3FileSystem(
    endpoint_override=S3_ENDPOINT,
    access_key=S3_KEY, secret_key=S3_SECRET,
    scheme="http", region="us-east-1",
)
s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                  aws_access_key_id=S3_KEY, aws_secret_access_key=S3_SECRET,
                  region_name="us-east-1")
print("clients ready")

clients ready


## 2. Клиент Trino с поддержкой session-свойств

In [2]:
def trino(sql, catalog="lakekeeper", schema="lakehouse", session=None, timeout=900):
    h = {"X-Trino-User": "trino", "X-Trino-Catalog": catalog, "X-Trino-Schema": schema}
    if session:
        h["X-Trino-Session"] = ",".join(f"{k}={v}" for k, v in session.items())
    r = requests.post(TRINO_URL + "/v1/statement", data=sql.encode(), headers=h, timeout=timeout)
    j = r.json()
    if j.get("error"):
        raise RuntimeError(j["error"].get("message", str(j)))
    cols = [c["name"] for c in j.get("columns", [])]
    data = j.get("data") or []
    uri = j.get("nextUri")
    while uri:
        j = requests.get(uri, headers={"X-Trino-User": "trino"}, timeout=timeout).json()
        if j.get("error"):
            raise RuntimeError(j["error"].get("message", str(j)))
        data += j.get("data") or []
        uri = j.get("nextUri")
    return cols, data

def trino_time(sql, **kw):
    t0 = time.perf_counter()
    trino(sql, **kw)
    return time.perf_counter() - t0

print("trino client ready")

trino client ready


## 3. Датасет (~3 млн строк) локально в памяти

In [3]:
rng = np.random.default_rng(42)
N = 3_000_000
regions = ["Москва", "Санкт-Петербург", "Казань", "Екатеринбург", "Новосибирск", "Сочи", "Краснодар"]
categories = ["Электроника", "Одежда", "Продукты", "Дом", "Спорт", "Книги", "Игрушки"]

df = pd.DataFrame({
    "order_id":   np.arange(N, dtype=np.int64),
    "store_id":   rng.integers(1, 200, N, dtype=np.int64),
    "region":     rng.choice(regions, N).astype(object),
    "category":   rng.choice(categories, N).astype(object),
    "price":      np.round(rng.lognormal(mean=4.0, sigma=1.0, size=N), 2),
    "qty":        rng.integers(1, 10, N, dtype=np.int32),
    "order_date": pd.date_range("2024-01-01", periods=N, freq="15s").strftime("%Y-%m-%d %H:%M:%S"),
    "is_promo":   rng.integers(0, 2, N).astype(bool),
})

tbl = pa.Table.from_pandas(df, preserve_index=False)
print("rows:", tbl.num_rows)
print("schema:")
print(tbl.schema)

rows: 3000000
schema:
order_id: int64
store_id: int64
region: large_string
category: large_string
price: double
qty: int32
order_date: large_string
is_promo: bool
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 980


## 4. Локальный бенчмарк Parquet vs ORC

`GZIP` в ORC у PyArrow называется `ZLIB` (тот же алгоритм deflate). Учти это при заполнении столбца «Сжатие».

In [4]:
import os, gc, time, uuid, shutil

# очистка папки bench от старых прогонов
if os.path.exists("bench"):
    shutil.rmtree("bench", ignore_errors=True)
os.makedirs("bench", exist_ok=True)

run_id = uuid.uuid4().hex[:6]   # уникальный суффикс, чтобы Windows не лочил файлы

parquet_codecs = ["NONE", "SNAPPY", "LZ4", "ZSTD", "GZIP"]
orc_codecs     = ["UNCOMPRESSED", "SNAPPY", "LZ4", "ZSTD", "ZLIB"]

rows = []

for c in parquet_codecs:
    p = f"bench/p_{c}_{run_id}.parquet"
    kw = {"compression": c}
    if c == "ZSTD": kw["compression_level"] = 3
    if c == "GZIP": kw["compression_level"] = 9
    t0 = time.perf_counter()
    pq.write_table(tbl, p, **kw)
    wt = time.perf_counter() - t0
    size = os.path.getsize(p)
    t0 = time.perf_counter()
    _t = pq.read_table(p)
    rt = time.perf_counter() - t0
    del _t
    gc.collect()
    rows.append(("Parquet", c, size, wt, rt))
    print(f"Parquet {c:10s} {size/1e6:8.2f} MB  write={wt:.2f}s  read={rt:.2f}s")

for c in orc_codecs:
    p = f"bench/o_{c}_{run_id}.orc"
    t0 = time.perf_counter()
    orc.write_table(tbl, p, compression=c)
    wt = time.perf_counter() - t0
    size = os.path.getsize(p)
    t0 = time.perf_counter()
    _t = orc.read_table(p)
    rt = time.perf_counter() - t0
    del _t
    gc.collect()
    rows.append(("ORC", c, size, wt, rt))
    print(f"ORC     {c:10s} {size/1e6:8.2f} MB  write={wt:.2f}s  read={rt:.2f}s")

local = pd.DataFrame(rows, columns=["format", "codec", "size_bytes", "write_s", "read_s"])
local["size_mb"] = (local["size_bytes"] / 1e6).round(2)
local["size_gb"] = (local["size_bytes"] / 1e9).round(4)
local = local[["format", "codec", "size_gb", "size_mb", "write_s", "read_s"]]
local

Parquet NONE         108.61 MB  write=1.20s  read=0.16s
Parquet SNAPPY        40.31 MB  write=1.12s  read=0.16s
Parquet LZ4           42.65 MB  write=1.31s  read=0.13s
Parquet ZSTD          18.67 MB  write=1.38s  read=0.13s
Parquet GZIP          25.77 MB  write=30.01s  read=0.13s
ORC     UNCOMPRESSED   188.82 MB  write=1.20s  read=0.40s
ORC     SNAPPY        54.41 MB  write=1.47s  read=0.47s
ORC     LZ4          188.83 MB  write=1.26s  read=0.42s
ORC     ZSTD          28.08 MB  write=1.65s  read=0.58s
ORC     ZLIB          34.03 MB  write=2.57s  read=0.67s


,format,codec,size_gb,size_mb,write_s,read_s
0,Parquet,NONE,0.1086,108.61,1.204822,0.155442
1,Parquet,SNAPPY,0.0403,40.31,1.120434,0.160903
2,Parquet,LZ4,0.0427,42.65,1.312989,0.126508
3,Parquet,ZSTD,0.0187,18.67,1.381052,0.130967
4,Parquet,GZIP,0.0258,25.77,30.013439,0.131859
5,ORC,UNCOMPRESSED,0.1888,188.82,1.200140,0.403680
6,ORC,SNAPPY,0.0544,54.41,1.469609,0.466815
7,ORC,LZ4,0.1888,188.83,1.258454,0.418500
8,ORC,ZSTD,0.0281,28.08,1.646960,0.577753
9,ORC,ZLIB,0.0340,34.03,2.574278,0.672207


## 5. Запись в MinIO (bucket `warehouse`) и подсчёт файлов

In [5]:
import glob

minio_rows = []
for src in sorted(glob.glob("bench/*.parquet")) + sorted(glob.glob("bench/*.orc")):
    base = os.path.basename(src)
    ext = base.rsplit(".", 1)[-1]           # parquet / orc
    fmt = "Parquet" if ext == "parquet" else "ORC"
    # имя в бакете: с уникальным суффиксом, чтобы не перезаписывать
    key = f"bench/{base}"
    s3.upload_file(src, "warehouse", key)
    minio_rows.append((fmt, base, key))
    print(f"uploaded {src} -> s3://warehouse/{key}")

print()
print("файлы в бакете warehouse/bench:")
for o in s3.list_objects_v2(Bucket="warehouse", Prefix="bench/").get("Contents", []):
    print(f"  {o['Key']:40s} {o['Size']/1e6:8.2f} MB")

uploaded bench\p_GZIP_a6f4aa.parquet -> s3://warehouse/bench/p_GZIP_a6f4aa.parquet
uploaded bench\p_LZ4_a6f4aa.parquet -> s3://warehouse/bench/p_LZ4_a6f4aa.parquet
uploaded bench\p_NONE_a6f4aa.parquet -> s3://warehouse/bench/p_NONE_a6f4aa.parquet
uploaded bench\p_SNAPPY_a6f4aa.parquet -> s3://warehouse/bench/p_SNAPPY_a6f4aa.parquet
uploaded bench\p_ZSTD_a6f4aa.parquet -> s3://warehouse/bench/p_ZSTD_a6f4aa.parquet
uploaded bench\o_LZ4_a6f4aa.orc -> s3://warehouse/bench/o_LZ4_a6f4aa.orc
uploaded bench\o_SNAPPY_a6f4aa.orc -> s3://warehouse/bench/o_SNAPPY_a6f4aa.orc
uploaded bench\o_UNCOMPRESSED_a6f4aa.orc -> s3://warehouse/bench/o_UNCOMPRESSED_a6f4aa.orc
uploaded bench\o_ZLIB_a6f4aa.orc -> s3://warehouse/bench/o_ZLIB_a6f4aa.orc
uploaded bench\o_ZSTD_a6f4aa.orc -> s3://warehouse/bench/o_ZSTD_a6f4aa.orc

файлы в бакете warehouse/bench:
  bench/o_LZ4_27eb08.orc                     188.83 MB
  bench/o_LZ4_55c9c1.orc                     188.83 MB
  bench/o_LZ4_6de25b.orc                     18

## 6. Iceberg в Trino: Parquet и ORC, разные кодеки

Trino поддерживает session-свойство `iceberg.compression_codec`. Ставим его **перед каждой вставкой** — он влияет на записываемые файлы.

In [6]:
trino('CREATE SCHEMA IF NOT EXISTS lakekeeper.lakehouse')

# проверим, что tpcds-коннектор виден
cols, data = trino("SHOW SCHEMAS FROM tpcds")
print("tpcds schemas:", [row[0] for row in data])

# сколько строк в store_sales sf1
cols, data = trino("SELECT count(*) FROM tpcds.sf1.store_sales")
print("store_sales sf1 rows:", data[0][0])

tpcds schemas: ['information_schema', 'sf1', 'sf10', 'sf100', 'sf1000', 'sf10000', 'sf100000', 'sf300', 'sf3000', 'sf30000', 'tiny']
store_sales sf1 rows: 2880404


## 7. Прогон по форматам и кодекам

In [7]:
trino_rows = []

DDL_ORC = '''CREATE TABLE lakekeeper.lakehouse.bench_tpcds (
  ss_sold_date_sk bigint, ss_sold_time_sk bigint, ss_item_sk bigint,
  ss_customer_sk bigint, ss_cdemo_sk bigint, ss_hdemo_sk bigint,
  ss_addr_sk bigint, ss_store_sk bigint, ss_promo_sk bigint,
  ss_ticket_number bigint, ss_quantity integer, ss_wholesale_cost double,
  ss_list_price double, ss_sales_price double, ss_ext_discount_amt double,
  ss_ext_sales_price double, ss_ext_wholesale_cost double, ss_ext_list_price double,
  ss_ext_tax double, ss_coupon_amt double, ss_net_paid double,
  ss_net_paid_inc_tax double, ss_net_profit double
) WITH (format = 'ORC')'''

trino("DROP TABLE IF EXISTS lakekeeper.lakehouse.bench_tpcds")
trino(DDL_ORC)

sess = {"lakekeeper.compression_codec": "ZSTD"}
t0 = time.perf_counter()
trino("INSERT INTO lakekeeper.lakehouse.bench_tpcds SELECT * FROM tpcds.sf1.store_sales", session=sess)
write_s = time.perf_counter() - t0

_, cnt = trino("SELECT count(*) FROM lakekeeper.lakehouse.bench_tpcds")
rows_in_table = int(cnt[0][0])

files, total = 0, 0
_, fdata = trino('SELECT file_path, file_size_in_bytes FROM lakekeeper.lakehouse."bench_tpcds$files"')
for fp, sz in fdata:
    files += 1
    total += int(sz)

# агрегации: три прогона
agg_sql = """
SELECT ss_store_sk, count(*) n, sum(ss_net_paid) s
FROM lakekeeper.lakehouse.bench_tpcds
GROUP BY ss_store_sk
"""
t_first = trino_time(agg_sql)
t_second = trino_time(agg_sql)
t_third = trino_time(agg_sql)

trino_rows.append({
    "format": "ORC", "codec": "ZSTD",
    "rows": rows_in_table,
    "size_bytes": total, "files": files,
    "write_s": round(write_s, 2),
    "agg_first_s": round(t_first, 3),
    "agg_next_s": round((t_second + t_third) / 2, 3),
})
print(f"rows={rows_in_table}, size={total/1e9:.4f} GB ({total/1e6:.1f} MB), files={files}")
print(f"write={write_s:.2f}s, agg1={t_first:.3f}s, agg2+={((t_second+t_third)/2):.3f}s")

iceberg = pd.DataFrame(trino_rows)
iceberg

rows=2880404, size=0.1195 GB (119.5 MB), files=1
write=16.49s, agg1=0.379s, agg2+=0.191s


,format,codec,rows,size_bytes,files,write_s,agg_first_s,agg_next_s
0,ORC,ZSTD,2880404,119483344,1,16.49,0.379,0.191


In [8]:
out = iceberg.copy()
out["Размер, ГБ"] = (out["size_bytes"] / 1e9).round(4)
out["Сжатие"] = out["codec"]
out["формат"] = out["format"]
out["Файлов"] = out["files"]
out["Время записи, с"] = out["write_s"]
out["Агрегация первая, сек"] = out["agg_first_s"]
out["Агрегация вторая и следующие, сек"] = out["agg_next_s"]
out["Trino"] = "OK"
out["spark"] = "—"

result = out[["Сжатие","формат","Размер, ГБ","Файлов","Время записи, с",
              "Агрегация первая, сек","Агрегация вторая и следующие, сек",
              "Trino","spark"]]
display(result)
print()
print("Скопируй это в свою Excel-таблицу.")
result.to_csv("result_lakehouse.csv", index=False)
print("также сохранено в result_lakehouse.csv")

,Сжатие,формат,"Размер, ГБ",Файлов,"Время записи, с","Агрегация первая, сек","Агрегация вторая и следующие, сек",Trino,spark
0,ZSTD,ORC,0.1195,1,16.49,0.379,0.191,OK,—



Скопируй это в свою Excel-таблицу.
также сохранено в result_lakehouse.csv


## 9. Локальный бенчмарк — вторая таблица (для наглядности)

In [9]:
display(local)
local.to_csv("result_local_bench.csv", index=False)
print("сохранено в result_local_bench.csv")

,format,codec,size_gb,size_mb,write_s,read_s
0,Parquet,NONE,0.1086,108.61,1.204822,0.155442
1,Parquet,SNAPPY,0.0403,40.31,1.120434,0.160903
2,Parquet,LZ4,0.0427,42.65,1.312989,0.126508
3,Parquet,ZSTD,0.0187,18.67,1.381052,0.130967
4,Parquet,GZIP,0.0258,25.77,30.013439,0.131859
5,ORC,UNCOMPRESSED,0.1888,188.82,1.200140,0.403680
6,ORC,SNAPPY,0.0544,54.41,1.469609,0.466815
7,ORC,LZ4,0.1888,188.83,1.258454,0.418500
8,ORC,ZSTD,0.0281,28.08,1.646960,0.577753
9,ORC,ZLIB,0.0340,34.03,2.574278,0.672207


сохранено в result_local_bench.csv
